# Notebook 19 — HTTP clients & timeouts (`httpx`)

| | |
|---|---|
| **Track** | AI Engineering · `03-python-foundations/exercises/` |
| **Level** | Intermediate → Advanced |
| **Pattern** | *Concept* → *Runnable demo* → *Your turn* → *Solution* |

**Prerequisites:** `18-asyncio-queue-pipelines.ipynb`

**Next up:** `20-positional-keyword-only-signatures.ipynb`

---

## Learning objectives

- Issue GET requests with explicit timeouts — LLM gateways punish hangs.
- Surface HTTP failures as actionable exceptions.
- Mirror FastAPI-facing discipline before mixing streaming APIs.

## Table of contents

1. **Install **`httpx`****
2. **Sync client + JSON payloads**
3. **`raise_for_status` semantics**
4. **Progressive drills — HEAD probe → JSON decode → typed error buckets**
5. **Exercise — `fetch_json` helper**

---

## How to use this notebook

1. Run cells **top to bottom** the first time through.
2. Re-run and **change values** to test your understanding.
3. Do **Your turn** cells before opening solutions (HTML `<details>` blocks or later cells).

---


### Dependency

```bash
pip install httpx
```

Requires outbound HTTPS for demo URLs (firewall-friendly).


## 1 · Sync client + JSON payloads

*Explanation:* **`httpx.Client`** composes defaults (`timeout`, headers). Swap for **`AsyncClient`** once **`await`** plumbing matches **`nb09`**.


In [ ]:
import httpx

sample_url = "https://jsonplaceholder.typicode.com/posts/1"

with httpx.Client(timeout=10.0) as client:
    response = client.get(sample_url)
    response.raise_for_status()
    payload = response.json()

print(payload["title"][:48])


## 2 · Error surfaces

*Explanation:* **`raise_for_status()`** maps failure bands—mirror later mappings from vendor codes to retries.


In [ ]:
import httpx

bad_url = "https://jsonplaceholder.typicode.com/posts/999999"

try:
    with httpx.Client(timeout=10.0) as client:
        r = client.get(bad_url)
        r.raise_for_status()
except httpx.HTTPStatusError as exc:
    print("HTTP layer:", exc.response.status_code)


## 3 · Headers & tracing placeholders

*Explanation:* Attach **`Authorization`** / **`x-request-id`** at construction time — avoids scattering secrets.


In [ ]:
import httpx

headers = {"User-Agent": "ai-foundations-drill/1.0", "x-demo": "trace-local"}

with httpx.Client(timeout=5.0, headers=headers) as client:
    r = client.get("https://jsonplaceholder.typicode.com/posts/2")
    r.raise_for_status()
    print(r.json()["id"])


---

## Progressive drills — **easy → harder**

**Tool routers** wrap HTTP boundaries — drills rehearse timeouts before wiring multimodal endpoints.


### A · Easiest — bounded latency

Never omit **`timeout=`** — infinite waits stall notebooks and workers alike.


In [ ]:
import httpx

with httpx.Client(timeout=1.0) as client:
    ok = client.get("https://jsonplaceholder.typicode.com/posts/3")
    ok.raise_for_status()
    print(ok.status_code)


### B · Medium — decode JSON safely

Isolate **`JSONDecodeError`** from transport failures.


In [ ]:
import httpx

with httpx.Client(timeout=5.0) as client:
    r = client.get("https://jsonplaceholder.typicode.com/posts/4")
    r.raise_for_status()
    body = r.json()
    assert isinstance(body, dict)
    print(sorted(body.keys())[:4])


### C · Harder — classify transport vs HTTP vs decode

Catch **`httpx.RequestError`** separately from status failures.


In [ ]:
import httpx

def classify(url: str) -> str:
    try:
        with httpx.Client(timeout=3.0) as client:
            r = client.get(url)
            r.raise_for_status()
    except httpx.HTTPStatusError:
        return "http_status"
    except httpx.RequestError:
        return "transport"
    return "ok"


print(classify("https://jsonplaceholder.typicode.com/posts/1"))
print(classify("https://jsonplaceholder.typicode.com/posts/40404"))


### Exercise — `fetch_json`

Implement **`fetch_json(url: str, *, timeout: float = 10.0) -> dict`** using **`httpx.Client(timeout=timeout)`**, **`GET`**, **`raise_for_status()`**, then **`response.json()`**. If decoding yields **`list`** instead of **`dict`**, raise **`TypeError("expected JSON object")`**.

Run against **`https://jsonplaceholder.typicode.com/posts/5`**.


In [ ]:
import httpx


def fetch_json(url: str, *, timeout: float = 10.0) -> dict:
    raise NotImplementedError


demo_url = "https://jsonplaceholder.typicode.com/posts/5"
payload = fetch_json(demo_url)
assert payload["id"] == 5
print("OK")


### Solution — fetch_json

<details>
<summary>Click to expand</summary>

```python
import httpx

def fetch_json(url: str, *, timeout: float = 10.0) -> dict:
    with httpx.Client(timeout=timeout) as client:
        response = client.get(url)
        response.raise_for_status()
        data = response.json()
        if not isinstance(data, dict):
            raise TypeError("expected JSON object")
        return data
```

</details>
